In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/playground-series-s5e6/sample_submission.csv
/kaggle/input/playground-series-s5e6/train.csv
/kaggle/input/playground-series-s5e6/test.csv
/kaggle/input/fertilizer-prediction/Fertilizer Prediction.csv


In [2]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("irakozekelly/fertilizer-prediction")

print("Path to dataset files:", path)

Path to dataset files: /kaggle/input/fertilizer-prediction


# Reference

- https://www.kaggle.com/code/elainedazzio/20250623-pg6-xgb PG6_XGB by L. Elaine Dazzio
- https://www.kaggle.com/code/ricopue/s5e6-fertilizers-nn-keras-all-embedding S5E6 Fertilizers - NN keras All Embedding by Ricardo Colomer
- https://www.kaggle.com/code/hahahaj/single-xgb by hahahaj
- https://www.kaggle.com/competitions/playground-series-s5e6/discussion/585000 Tips & Tricks to improve the score : Just Consider everything as Categorical Data by Gaurav Dutta

XGBoost + NN Ensemble

# 0. import libraries

In [3]:
import pandas as pd
import numpy as np
import random
import os
import gc
import joblib
import warnings
warnings.simplefilter('ignore')

# ML Libraries
import xgboost as xgb
from sklearn.model_selection import RepeatedStratifiedKFold, StratifiedKFold
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

# Deep Learning Libraries
import tensorflow as tf
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.layers import Input, Embedding, SpatialDropout1D, Reshape, Concatenate, Dropout, BatchNormalization, Dense
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import ReduceLROnPlateau, EarlyStopping
from tensorflow.keras.regularizers import l1

2025-06-25 05:48:26.559653: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1750830506.827819      35 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1750830506.903365      35 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


# 1. Configuration

In [4]:
# Configuration
class CFG:
    path = "../input/playground-series-s5e6/"
    original_path = "../input/fertilizer-prediction/"
    
    n_splits = 10
    n_repeats = 1
    seed = 42
    
    # XGBoost parameters
    learning_rate = 5e-2
    num_boost_round = 10000
    early_stopping_rounds = 100
    verbose_eval = 500
    
    # Neural Network parameters
    FOLDS = 5
    epochs = 100
    verbose = 1
    
    target = "Fertilizer_Name"  # Updated target name

def seed_everything(seed=42):
    """Set random seeds for reproducibility"""
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)

# 2. Data Load and Preprocessing

In [5]:
# Data Loading and Preprocessing
def load_and_preprocess_data():
    """Load and preprocess the dataset"""
    print("Loading data...")
    
    # Load data
    train = pd.read_csv(CFG.path + "train.csv").drop(columns=['id'])
    test = pd.read_csv(CFG.path + "test.csv").drop(columns=['id'])
    original = pd.read_csv(CFG.original_path + "Fertilizer Prediction.csv")
    
    # Augment original data
    original_copy = original.copy()
    for i in range(5):
        original = pd.concat([original, original_copy], ignore_index=True)
    
    # Fix column names consistently
    def rename_columns(df):
        df = df.rename(columns={
            'Temparature': 'Temperature',
            'Soil Type': 'Soil_Type', 
            'Crop Type': 'Crop_Type', 
            'Fertilizer Name': 'Fertilizer_Name'
        })
        return df
    
    train = rename_columns(train)
    test = rename_columns(test)
    original = rename_columns(original)
    
    print("Data loaded and column names standardized")
    print(f"Train columns: {train.columns.tolist()}")
    print(f"Test columns: {test.columns.tolist()}")
    
    return train, test, original

def prepare_xgb_features(train, test, original):
    """Prepare features for XGBoost model"""
    print("Preparing XGBoost features...")
    
    # Add binned categorical features for numerical columns
    num_cols = [col for col in test.select_dtypes(exclude=['object', 'category']).columns]
    
    def add_categorical_features(df, cols):
        for col in cols:
            df[f"{col}_Binned"] = df[col].astype("category")
        return df
    
    train_xgb = add_categorical_features(train.copy(), num_cols)
    test_xgb = add_categorical_features(test.copy(), num_cols)
    original_xgb = add_categorical_features(original.copy(), num_cols)
    
    # Encode categorical features for XGBoost
    cat_cols = [col for col in test_xgb.select_dtypes(include=['object', 'category']).columns]
    
    for col in cat_cols:
        label_enc = LabelEncoder()
        train_xgb[col] = label_enc.fit_transform(train_xgb[col])
        test_xgb[col] = label_enc.transform(test_xgb[col])
        original_xgb[col] = label_enc.transform(original_xgb[col])
        
        # Convert to category type for XGBoost
        train_xgb[col] = train_xgb[col].astype("category")
        test_xgb[col] = test_xgb[col].astype("category")
        original_xgb[col] = original_xgb[col].astype("category")
    
    # Encode target variable
    target_label_enc = LabelEncoder()
    train_xgb[CFG.target] = target_label_enc.fit_transform(train_xgb[CFG.target])
    original_xgb[CFG.target] = target_label_enc.transform(original_xgb[CFG.target])
    
    # Prepare feature list
    features = [col for col in test_xgb.columns if col != CFG.target]
    
    # Remove constant features
    constant_features = []
    for col in features:
        if train_xgb[col].nunique() == 1:
            constant_features.append(col)
    
    features = [col for col in features if col not in constant_features]
    
    if constant_features:
        print(f"Removed constant features: {constant_features}")
    
    print(f"XGBoost features: {features}")
    
    return train_xgb, test_xgb, original_xgb, features, target_label_enc

def prepare_nn_features(train, test, original):
    """Prepare features for Neural Network model"""
    print("Preparing Neural Network features...")
    
    # Use core features for NN
    nn_features = ['Temperature', 'Humidity', 'Moisture', 'Soil_Type', 'Crop_Type', 
                   'Nitrogen', 'Potassium', 'Phosphorous']
    
    # Check if all features exist
    missing_features = [f for f in nn_features if f not in train.columns]
    if missing_features:
        print(f"Missing features: {missing_features}")
        nn_features = [f for f in nn_features if f in train.columns]
    
    print(f"NN features: {nn_features}")
    
    # Create copies for NN processing
    train_nn = train[nn_features + [CFG.target]].copy()
    test_nn = test[nn_features].copy()
    original_nn = original[nn_features + [CFG.target]].copy()
    
    # Create encoding dictionaries
    Fertilizer_Name_dict_inv = dict(enumerate(train_nn[CFG.target].unique()))
    Fertilizer_Name_dict = {v: k for k, v in Fertilizer_Name_dict_inv.items()}
    
    Soil_Type_dict_inv = dict(enumerate(train_nn['Soil_Type'].unique()))
    Soil_Type_dict = {v: k for k, v in Soil_Type_dict_inv.items()}
    
    Crop_Type_dict_inv = dict(enumerate(train_nn['Crop_Type'].unique()))
    Crop_Type_dict = {v: k for k, v in Crop_Type_dict_inv.items()}
    
    print(f"Fertilizer_Name_dict: {Fertilizer_Name_dict}")
    print(f"Soil_Type_dict: {Soil_Type_dict}")
    print(f"Crop_Type_dict: {Crop_Type_dict}")
    
    # Combine train and test for consistent preprocessing
    df_combi = pd.concat([train_nn, test_nn]).reset_index(drop=True)
    
    # Apply categorical mappings
    df_combi['Soil_Type'] = df_combi['Soil_Type'].map(Soil_Type_dict)
    df_combi['Crop_Type'] = df_combi['Crop_Type'].map(Crop_Type_dict)
    df_combi['Soil_Type'] = df_combi['Soil_Type'].astype('int32')
    df_combi['Crop_Type'] = df_combi['Crop_Type'].astype('int32')
    
    # Normalize numerical features
    numerical_features = ['Temperature', 'Humidity', 'Moisture', 'Nitrogen', 'Potassium', 'Phosphorous']
    for col in numerical_features:
        if col in df_combi.columns:
            df_combi[col] = (df_combi[col] - df_combi[col].min()).astype('int32')
    
    # Split back
    train_nn_processed = df_combi[:len(train_nn)]
    test_nn_processed = df_combi[len(train_nn):]
    
    # Apply target encoding to train
    train_nn_processed[CFG.target] = train_nn_processed[CFG.target].map(Fertilizer_Name_dict)
    
    return train_nn_processed, test_nn_processed, nn_features, Fertilizer_Name_dict

# 3. Evaluation Metrics

In [6]:
# Evaluation Metrics
def mapk(actual, predicted, k=3):
    """Mean Average Precision at K"""
    def apk(a, p, k):
        p = p[:k]
        score = 0.0
        hits = 0
        seen = set()
        for i, pred in enumerate(p):
            if pred in a and pred not in seen:
                hits += 1
                score += hits / (i + 1.0)
                seen.add(pred)
        return score / min(len(a), k)
    return np.mean([apk(a, p, k) for a, p in zip(actual, predicted)])

def top_3_accuracy(y_true, y_pred):
    """Custom metric for Neural Network"""
    dd = tf.keras.metrics.top_k_categorical_accuracy(y_true, y_pred, k=1) * 0.5
    dd += tf.keras.metrics.top_k_categorical_accuracy(y_true, y_pred, k=2) * 0.17
    dd += tf.keras.metrics.top_k_categorical_accuracy(y_true, y_pred, k=3) * 0.33
    return dd

# 4. Model Training

## XGBoost

In [14]:
# XGBoost Model
def train_xgboost(train_xgb, test_xgb, original_xgb, features, target_label_enc):
    """Train XGBoost model with cross-validation"""
    print("Training XGBoost model...")
    
    # XGBoost parameters
    params = {
        'objective': 'multi:softprob',
        'num_class': train_xgb[CFG.target].nunique(),
        'seed': CFG.seed,
        'max_depth': 8,
        'max_bins': 128,
        'learning_rate': CFG.learning_rate,
        'min_child_weight': 2,
        'alpha': 6.5,
        'reg_lambda': 5.3,
        'subsample': 0.8,
        'colsample_bytree': 0.3,
        'tree_method': 'hist',
        'device': "gpu"
    }
    
    # Initialize predictions
    oof = np.zeros([train_xgb.shape[0], train_xgb[CFG.target].nunique()])
    pred = np.zeros([test_xgb.shape[0], train_xgb[CFG.target].nunique()])
    
    # Cross-validation
    kf = RepeatedStratifiedKFold(n_splits=CFG.n_splits, n_repeats=CFG.n_repeats, 
                                random_state=CFG.seed)
    
    for fold, (trn_idx, val_idx) in enumerate(kf.split(train_xgb, train_xgb[CFG.target])):
        print(f"Fold {fold + 1}/{CFG.n_splits * CFG.n_repeats}")
        
        X_train = train_xgb.loc[trn_idx, features]
        y_train = train_xgb.loc[trn_idx, CFG.target]
        X_valid = train_xgb.loc[val_idx, features]
        y_valid = train_xgb.loc[val_idx, CFG.target]
        X_test = test_xgb[features].copy()
        
        # Add original data to training set
        X_train = pd.concat([X_train, original_xgb[features]], ignore_index=True)
        y_train = pd.concat([y_train, original_xgb[CFG.target]], ignore_index=True)
        
        # Create DMatrix for XGBoost
        dtrain = xgb.DMatrix(X_train, label=y_train, enable_categorical=True)
        dvalid = xgb.DMatrix(X_valid, label=y_valid, enable_categorical=True)
        dtest = xgb.DMatrix(X_test, enable_categorical=True)
        
        # Early stopping callback
        ES = xgb.callback.EarlyStopping(
            rounds=CFG.early_stopping_rounds,
            maximize=False,
            save_best=True,
        )
        
        # Train model
        model = xgb.train(
            params, 
            dtrain, 
            num_boost_round=CFG.num_boost_round, 
            evals=[(dtrain, 'train'), (dvalid, 'validation')], 
            verbose_eval=CFG.verbose_eval,
            callbacks=[ES]
        )
        
        # Predictions
        tmp_oof = model.predict(dvalid)
        oof[val_idx] += tmp_oof / CFG.n_repeats
        pred += (model.predict(dtest) / (CFG.n_splits * CFG.n_repeats))
        
        # Calculate MAP@3
        top3_preds = np.argsort(tmp_oof, axis=1)[:, -3:][:, ::-1]
        actual = [[label] for label in y_valid]
        map3_score = mapk(actual, top3_preds)
        
        print(f"Fold {fold + 1} MAP@3: {map3_score:.6f}")
        print("-" * 50)
    
    # Overall OOF score
    top3_preds = np.argsort(oof, axis=1)[:, -3:][:, ::-1]
    actual = [[label] for label in train_xgb[CFG.target]]
    oof_map3 = mapk(actual, top3_preds)
    print(f"XGBoost OOF MAP@3: {oof_map3:.6f}")
    
    return oof, pred

## NN

In [8]:
# Neural Network Model
def preproc_nn(X_train, X_val, X_test, features):
    """Preprocess data for Neural Network"""
    input_list_train = []
    input_list_val = []
    input_list_test = []
    
    categoricals = ['int32']
    
    for c in X_train.select_dtypes(include=categoricals):
        raw_vals = np.unique(X_train[c])
        val_map = {}
        for i in range(len(raw_vals)):
            val_map[raw_vals[i]] = i
        
        input_list_train.append(X_train[c].map(val_map).values)
        input_list_val.append(X_val[c].map(val_map).fillna(0).values)
        input_list_test.append(X_test[c].map(val_map).fillna(0).values)
    
    return input_list_train, input_list_val, input_list_test

def create_nn_model(train, features, N_CLASSES):
    """Create Neural Network model"""
    input_models = []
    output_embeddings = []
    
    for categorical_var in features:
        cat_emb_name = categorical_var.replace(" ", "") + '_Embedding'
        no_of_unique_cat = train[categorical_var].nunique()
        embedding_size = int(min(np.ceil((no_of_unique_cat * 2)), 64))
        
        input_model = Input(shape=(1,), name=categorical_var)
        output_model = Embedding(no_of_unique_cat, embedding_size, name=cat_emb_name)(input_model)
        
        if embedding_size > 25:
            output_model = SpatialDropout1D(0.2)(output_model)
        
        output_model = Reshape(target_shape=(embedding_size,))(output_model)
        
        input_models.append(input_model)
        output_embeddings.append(output_model)
    
    # Concatenate and add dense layers
    output = Concatenate()(output_embeddings)
    output = Dropout(0.2)(output)
    output = BatchNormalization()(output)
    
    output = Dense(1024, activation='relu', kernel_initializer="uniform")(output)
    output = Dropout(0.2)(output)
    output = BatchNormalization()(output)
    
    output = Dense(128, activation='relu', kernel_initializer="uniform")(output)
    output = Dropout(0.2)(output)
    output = BatchNormalization()(output)
    
    output = Dense(N_CLASSES, activation="softmax", kernel_regularizer=l1(1e-4))(output)
    
    model = Model(inputs=input_models, outputs=output)
    model.compile(
        optimizer=Adam(learning_rate=0.0002),
        loss='categorical_crossentropy',
        metrics=[top_3_accuracy]
    )
    
    return model

def train_neural_network(train_nn, test_nn, features, Fertilizer_Name_dict):
    """Train Neural Network model"""
    print("Training Neural Network model...")
    
    # Prepare target
    y_ori = train_nn[CFG.target].values
    y = to_categorical(train_nn[CFG.target])
    N_CLASSES = len(Fertilizer_Name_dict)
    
    # Remove target from features
    train_nn_features = train_nn.drop([CFG.target], axis=1)
    
    # Initialize predictions
    train_oof = np.zeros((len(train_nn_features), N_CLASSES))
    test_pred = np.zeros((len(test_nn), N_CLASSES))
    
    # Cross-validation
    kf = StratifiedKFold(n_splits=CFG.FOLDS, shuffle=True, random_state=42)
    
    for i, (train_index, valid_index) in enumerate(kf.split(train_nn_features, y_ori)):
        print(f"Training Fold {i + 1}/{CFG.FOLDS}...")
        
        X_tr = train_nn_features.loc[train_index, features]
        y_tr = y[train_index]
        X_val = train_nn_features.loc[valid_index, features]
        y_val = y[valid_index]
        
        X_train_list, X_val_list, X_test_list = preproc_nn(X_tr, X_val, test_nn[features], features)
        
        # Clear session and create model
        tf.keras.backend.clear_session()
        model = create_nn_model(train_nn_features, features, N_CLASSES)
        
        # Callbacks
        lr_callback = ReduceLROnPlateau(
            monitor='val_top_3_accuracy',
            factor=0.5,
            patience=1,
            verbose=CFG.verbose,
            min_lr=1e-6,
            mode="max"
        )
        
        early_stop = EarlyStopping(
            monitor='val_top_3_accuracy',
            patience=5,
            restore_best_weights=True,
            mode="max",
            verbose=CFG.verbose
        )
        
        # Train model
        history = model.fit(
            X_train_list, y_tr,
            validation_data=(X_val_list, y_val),
            verbose=CFG.verbose,
            epochs=CFG.epochs,
            batch_size=1024 * 2,
            callbacks=[lr_callback, early_stop]
        )
        
        # Predictions
        oof_preds = model.predict(X_val_list, verbose=0, batch_size=1024 * 8)
        train_oof[valid_index] = oof_preds
        test_pred += model.predict(X_test_list, verbose=0, batch_size=1024 * 8)
        
        # Calculate fold score
        oof_map = top_3_accuracy(
            tf.convert_to_tensor(y_val, dtype=tf.float32),
            tf.convert_to_tensor(oof_preds, dtype=tf.float32)
        ).numpy().mean()
        
        print(f"Fold {i + 1} MAP3 score: {oof_map:.6f}")
    
    # Average test predictions
    test_pred = test_pred / CFG.FOLDS
    
    # Overall OOF score
    oof_map = top_3_accuracy(
        tf.convert_to_tensor(y, dtype=tf.float32),
        tf.convert_to_tensor(train_oof, dtype=tf.float32)
    ).numpy().mean()
    
    print(f"Neural Network OOF MAP3: {oof_map:.6f}")
    
    return train_oof, test_pred

# 5. Ensemble

In [17]:
# Ensemble Model
class Trainer:
    """Trainer class for ensemble model"""
    def __init__(self, model):
        self.model = model
    
    def fit_predict(self, X, y, X_test):
        """Fit model and make predictions"""
        # Fit model
        self.model.fit(X, y)
        
        # OOF predictions using cross-validation
        kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
        oof_preds = np.zeros(len(X))
        
        for train_idx, val_idx in kf.split(X, y):
            fold_model = LogisticRegression(
                random_state=42,
                max_iter=1000,
                solver='liblinear',
                penalty='l2',
                C=32.89802104596641,
                tol=0.0029878837974181643,
                fit_intercept=True
            )
            fold_model.fit(X.iloc[train_idx], y[train_idx])
            oof_preds[val_idx] = fold_model.predict(X.iloc[val_idx])
        
        # Test predictions
        test_preds = self.model.predict_proba(X_test)
        
        # Calculate score
        score = accuracy_score(y, oof_preds)
        
        return oof_preds, test_preds, score

def ensemble_models(xgb_oof, xgb_pred, nn_oof, nn_pred, y_true):
    """Ensemble XGBoost and Neural Network predictions"""
    print("Training ensemble model...")
    
    # Combine OOF predictions
    X = pd.DataFrame(np.concatenate([xgb_oof, nn_oof], axis=1))
    X_test = pd.DataFrame(np.concatenate([xgb_pred, nn_pred], axis=1))
    
    # Train ensemble model
    lr_model = LogisticRegression(
        random_state=42,
        max_iter=1000,
        solver='liblinear',
        penalty='l2',
        C=32.89802104596641,
        tol=0.0029878837974181643,
        fit_intercept=True
    )
    
    lr_trainer = Trainer(lr_model)
    _, lr_test_pred_probs, ensemble_score = lr_trainer.fit_predict(X, y_true, X_test)
    
    print(f"Ensemble accuracy: {ensemble_score:.6f}")
    
    return lr_test_pred_probs

# 6. Submission

In [33]:
# Separated execution functions
def run_data_preprocessing():
    """Execute data preprocessing only"""
    print("=== Data Preprocessing ===")
    seed_everything(CFG.seed)
    
    # Load and preprocess data
    train, test, original = load_and_preprocess_data()
    
    # Prepare features for both models
    train_xgb, test_xgb, original_xgb, xgb_features, target_label_enc = prepare_xgb_features(train, test, original)
    train_nn, test_nn, nn_features, Fertilizer_Name_dict = prepare_nn_features(train, test, original)
    
    print(f"XGBoost - Number of features: {len(xgb_features)}")
    print(f"NN - Number of features: {len(nn_features)}")
    print(f"Number of classes: {train_xgb[CFG.target].nunique()}")
    
    # Save preprocessed data
    joblib.dump(train_xgb, "train_xgb_processed.pkl")
    joblib.dump(test_xgb, "test_xgb_processed.pkl") 
    joblib.dump(original_xgb, "original_xgb_processed.pkl")
    joblib.dump(xgb_features, "xgb_features.pkl")
    joblib.dump(target_label_enc, "target_label_enc.pkl")
    
    joblib.dump(train_nn, "train_nn_processed.pkl")
    joblib.dump(test_nn, "test_nn_processed.pkl")
    joblib.dump(nn_features, "nn_features.pkl")
    joblib.dump(Fertilizer_Name_dict, "fertilizer_name_dict.pkl")
    
    print("Data preprocessing completed and saved!")
    return train_xgb, test_xgb, original_xgb, xgb_features, target_label_enc, train_nn, test_nn, nn_features, Fertilizer_Name_dict

def run_xgboost_training():
    """Train XGBoost model only"""
    print("=== XGBoost Training ===")
    
    # Load preprocessed data
    try:
        train_xgb = joblib.load("train_xgb_processed.pkl")
        test_xgb = joblib.load("test_xgb_processed.pkl")
        original_xgb = joblib.load("original_xgb_processed.pkl")
        xgb_features = joblib.load("xgb_features.pkl")
        target_label_enc = joblib.load("target_label_enc.pkl")
    except FileNotFoundError:
        print("Preprocessed data not found. Running preprocessing first...")
        train_xgb, test_xgb, original_xgb, xgb_features, target_label_enc, _, _, _, _ = run_data_preprocessing()
    
    # Train XGBoost
    xgb_oof, xgb_pred = train_xgboost(train_xgb, test_xgb, original_xgb, xgb_features, target_label_enc)
    
    # Save XGBoost predictions
    joblib.dump(xgb_oof, "xgb_oof.pkl")
    joblib.dump(xgb_pred, "xgb_pred.pkl")
    
    print("XGBoost training completed!")
    return xgb_oof, xgb_pred

def run_neural_network_training():
    """Train Neural Network model only"""
    print("=== Neural Network Training ===")
    
    # Load preprocessed data
    try:
        train_nn = joblib.load("train_nn_processed.pkl")
        test_nn = joblib.load("test_nn_processed.pkl")
        nn_features = joblib.load("nn_features.pkl")
        Fertilizer_Name_dict = joblib.load("fertilizer_name_dict.pkl")
    except FileNotFoundError:
        print("Preprocessed data not found. Running preprocessing first...")
        _, _, _, _, _, train_nn, test_nn, nn_features, Fertilizer_Name_dict = run_data_preprocessing()
    
    # Train Neural Network
    nn_oof, nn_pred = train_neural_network(train_nn, test_nn, nn_features, Fertilizer_Name_dict)
    
    # Save NN predictions
    joblib.dump(nn_oof, "nn_oof.pkl")
    joblib.dump(nn_pred, "nn_pred.pkl")
    
    print("Neural Network training completed!")
    return nn_oof, nn_pred

def run_ensemble(xgb_oof=None, xgb_pred=None, nn_oof=None, nn_pred=None, train_target=None, target_label_enc=None):
    """Train ensemble model and create final predictions"""
    print("=== Ensemble Training ===")
    
    # ... (기존 코드들) ...
    
    # 테스트 데이터의 원본 ID 로드
    try:
        # 원본 테스트 데이터에서 ID 가져오기
        test_data = pd.read_csv('test.csv')  # 또는 원본 테스트 파일 경로
        test_ids = test_data['id']
    except:
        try:
            # 또는 저장된 테스트 데이터에서 ID 가져오기
            test_processed = joblib.load("test_xgb_processed.pkl")
            if 'id' in test_processed.columns:
                test_ids = test_processed['id']
            else:
                print("ID column not found in processed test data")
                return None
        except:
            print("Cannot load test data IDs")
            return None
    
    # Ensemble models
    ensemble_pred = ensemble_models(xgb_oof, xgb_pred, nn_oof, nn_pred, train_target)
    
    # Create submission with correct IDs
    submission = pd.DataFrame({
        'id': test_ids,  # 실제 테스트 데이터의 ID 사용
        'Fertilizer Name': target_label_enc.inverse_transform(np.argmax(ensemble_pred, axis=1))
    })
    
    submission.to_csv('submission.csv', index=False)
    print("Submission file created: submission.csv")
    print("Ensemble training completed!")
    
    return ensemble_pred
    
# Main Pipeline
def main():
    """Main pipeline function - Execute complete pipeline"""
    print("Starting Complete Fertilizer Prediction Pipeline...")
    
    # 1. Data Preprocessing
    train_xgb, test_xgb, original_xgb, xgb_features, target_label_enc, train_nn, test_nn, nn_features, Fertilizer_Name_dict = run_data_preprocessing()
    
    # 2. XGBoost Training
    xgb_oof, xgb_pred = train_xgboost(train_xgb, test_xgb, original_xgb, xgb_features, target_label_enc)
    
    # 3. Neural Network Training
    nn_oof, nn_pred = train_neural_network(train_nn, test_nn, nn_features, Fertilizer_Name_dict)
    
    # 4. Ensemble Training
    ensemble_pred = run_ensemble()

In [11]:
# 1. Data Preprocessing
train_xgb, test_xgb, original_xgb, xgb_features, target_label_enc, train_nn, test_nn, nn_features, Fertilizer_Name_dict = run_data_preprocessing()

=== Data Preprocessing ===
Loading data...
Data loaded and column names standardized
Train columns: ['Temperature', 'Humidity', 'Moisture', 'Soil_Type', 'Crop_Type', 'Nitrogen', 'Potassium', 'Phosphorous', 'Fertilizer_Name']
Test columns: ['Temperature', 'Humidity', 'Moisture', 'Soil_Type', 'Crop_Type', 'Nitrogen', 'Potassium', 'Phosphorous']
Preparing XGBoost features...
XGBoost features: ['Temperature', 'Humidity', 'Moisture', 'Soil_Type', 'Crop_Type', 'Nitrogen', 'Potassium', 'Phosphorous', 'Temperature_Binned', 'Humidity_Binned', 'Moisture_Binned', 'Nitrogen_Binned', 'Potassium_Binned', 'Phosphorous_Binned']
Preparing Neural Network features...
NN features: ['Temperature', 'Humidity', 'Moisture', 'Soil_Type', 'Crop_Type', 'Nitrogen', 'Potassium', 'Phosphorous']
Fertilizer_Name_dict: {'28-28': 0, '17-17-17': 1, '10-26-26': 2, 'DAP': 3, '20-20': 4, '14-35-14': 5, 'Urea': 6}
Soil_Type_dict: {'Clayey': 0, 'Sandy': 1, 'Red': 2, 'Loamy': 3, 'Black': 4}
Crop_Type_dict: {'Sugarcane': 0, 'M

In [15]:
# 2. XGBoost Training
xgb_oof, xgb_pred = train_xgboost(train_xgb, test_xgb, original_xgb, xgb_features, target_label_enc)

Training XGBoost model...
Fold 1/10
[0]	train-mlogloss:1.94523	validation-mlogloss:1.94549
[500]	train-mlogloss:1.72287	validation-mlogloss:1.89092
[1000]	train-mlogloss:1.57586	validation-mlogloss:1.88209
[1166]	train-mlogloss:1.53475	validation-mlogloss:1.88208
Fold 1 MAP@3: 0.375407
--------------------------------------------------
Fold 2/10
[0]	train-mlogloss:1.94522	validation-mlogloss:1.94545
[500]	train-mlogloss:1.72333	validation-mlogloss:1.88991
[1000]	train-mlogloss:1.57610	validation-mlogloss:1.88034
[1208]	train-mlogloss:1.52569	validation-mlogloss:1.87994
Fold 2 MAP@3: 0.376240
--------------------------------------------------
Fold 3/10
[0]	train-mlogloss:1.94522	validation-mlogloss:1.94546
[500]	train-mlogloss:1.72322	validation-mlogloss:1.88953
[1000]	train-mlogloss:1.57599	validation-mlogloss:1.88019
[1221]	train-mlogloss:1.52260	validation-mlogloss:1.88023
Fold 3 MAP@3: 0.376569
--------------------------------------------------
Fold 4/10
[0]	train-mlogloss:1.94522	v

In [12]:
# 3. Neural Network Training
nn_oof, nn_pred = train_neural_network(train_nn, test_nn, nn_features, Fertilizer_Name_dict)

Training Neural Network model...
Training Fold 1/5...


I0000 00:00:1750830537.200344      35 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 15513 MB memory:  -> device: 0, name: Tesla P100-PCIE-16GB, pci bus id: 0000:00:04.0, compute capability: 6.0


Epoch 1/100


I0000 00:00:1750830545.459226      94 service.cc:148] XLA service 0x7da0840043a0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1750830545.460378      94 service.cc:156]   StreamExecutor device (0): Tesla P100-PCIE-16GB, Compute Capability 6.0
I0000 00:00:1750830546.093843      94 cuda_dnn.cc:529] Loaded cuDNN version 90300


 30/293 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - loss: 2.5327 - top_3_accuracy: 0.2643

I0000 00:00:1750830552.107591      94 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


293/293 ━━━━━━━━━━━━━━━━━━━━ 23s 33ms/step - loss: 2.3384 - top_3_accuracy: 0.2690 - val_loss: 1.9608 - val_top_3_accuracy: 0.2760 - learning_rate: 2.0000e-04
Epoch 2/100
293/293 ━━━━━━━━━━━━━━━━━━━━ 2s 6ms/step - loss: 2.0777 - top_3_accuracy: 0.2796 - val_loss: 1.9445 - val_top_3_accuracy: 0.2974 - learning_rate: 2.0000e-04
Epoch 3/100
293/293 ━━━━━━━━━━━━━━━━━━━━ 2s 6ms/step - loss: 2.0043 - top_3_accuracy: 0.2862 - val_loss: 1.9386 - val_top_3_accuracy: 0.3065 - learning_rate: 2.0000e-04
Epoch 4/100
293/293 ━━━━━━━━━━━━━━━━━━━━ 2s 6ms/step - loss: 1.9707 - top_3_accuracy: 0.2916 - val_loss: 1.9348 - val_top_3_accuracy: 0.3099 - learning_rate: 2.0000e-04
Epoch 5/100
293/293 ━━━━━━━━━━━━━━━━━━━━ 2s 6ms/step - loss: 1.9521 - top_3_accuracy: 0.2992 - val_loss: 1.9329 - val_top_3_accuracy: 0.3135 - learning_rate: 2.0000e-04
Epoch 6/100
293/293 ━━━━━━━━━━━━━━━━━━━━ 2s 6ms/step - loss: 1.9421 - top_3_accuracy: 0.3050 - val_loss: 1.9317 - val_top_3_accuracy: 0.3157 - learning_rate: 2.0000e

In [34]:
# 4. Ensemble Training
ensemble_pred = run_ensemble(xgb_oof, xgb_pred, nn_oof, nn_pred, train_xgb[CFG.target], target_label_enc)

=== Ensemble Training ===
ID column not found in processed test data


In [31]:
def evaluate_ensemble_map(ensemble_pred, train_target):
    """간단한 앙상블 MAP@3, MAP@5 평가"""
    
    # 데이터 형태 확인 및 디버깅
    print(f"Debug - ensemble_pred shape: {ensemble_pred.shape}")
    print(f"Debug - train_target shape: {train_target.shape}")
    
    def map_k(y_true, y_pred, k=3):
        # 배열 형태 확인
        if hasattr(y_true, 'values'):
            y_true = y_true.values
        if len(y_true) != len(y_pred):
            print(f"Error: Length mismatch - y_true: {len(y_true)}, y_pred: {len(y_pred)}")
            return 0.0
            
        map_scores = []
        for i in range(len(y_true)):
            true_label = y_true[i]
            pred_labels = np.argsort(y_pred[i])[::-1][:k]  # 상위 k개 예측
            
            if true_label in pred_labels:
                rank = np.where(pred_labels == true_label)[0][0] + 1
                map_scores.append(1.0 / rank)
            else:
                map_scores.append(0.0)
        return np.mean(map_scores)
    
    # MAP@3, MAP@5 계산
    map3 = map_k(train_target, ensemble_pred, k=3)
    map5 = map_k(train_target, ensemble_pred, k=5)
    
    print("=== Ensemble Performance ===")
    print(f"MAP@3: {map3:.6f}")
    print(f"MAP@5: {map5:.6f}")
    
    return map3, map5

# 사용법:
# map3, map5 = evaluate_ensemble_map(ensemble_pred, train_xgb[CFG.target])

In [32]:
map3, map5 = evaluate_ensemble_map(ensemble_pred, train_xgb[CFG.target])

Debug - ensemble_pred shape: (250000, 7)
Debug - train_target shape: (750000,)
Error: Length mismatch - y_true: 750000, y_pred: 250000
Error: Length mismatch - y_true: 750000, y_pred: 250000
=== Ensemble Performance ===
MAP@3: 0.000000
MAP@5: 0.000000


--- 
# FINISH